# Zn 배위 자리 채점 파이프라인**사용법:** 위에서부터 각 칸 왼쪽의 ▶ 버튼을 순서대로 누르면 됨.준비물: AF3에서 받은 zip 안의 `*.cif` 파일들, 그리고 `metal_site_score.py`, `site_function.py`, `renumber_site.py`, `zn_site.json`

## 1. 필요한 패키지 설치처음 한 번만. 1분쯤 걸림.

In [ ]:
!pip install -q gemmi numpyprint('설치 완료')

## 2. 파일 올리기▶ 누르면 파일 선택 창이 뜸. 아래를 **전부** 선택해서 올리기:- `metal_site_score.py`- `site_function.py`- `renumber_site.py`- `zn_site.json`- AF3 결과 `.cif` 파일들 (또는 `.pdb`)

In [ ]:
from google.colab import filesup = files.upload()print(f'\n올린 파일 {len(up)}개:')for k in up: print(' ', k)

## 3. cif → pdb 변환AF3는 `.cif`로 주는데 채점기는 `.pdb`를 읽음. `.pdb`만 올렸으면 이 칸은 건너뛰어도 됨.

In [ ]:
import gemmi, glob, oscifs = glob.glob('*.cif')for f in cifs:    st = gemmi.read_structure(f)    st.setup_entities()    out = f.replace('.cif', '.pdb')    st.write_pdb(out)    print(f'{f} → {out}')if not cifs:    print('cif 파일 없음 (pdb를 직접 올렸으면 정상)')

## 4. 잔기 번호 맞추기설계 파일은 잔기가 1..N으로 매겨져 있어서, 배위 His 3개를 찾아 94/96/119로 다시 붙임.**출력 확인:** `가상금속 잔차`가 0.5Å 넘으면 ⚠ 경고가 뜸. 그 파일은 배위 자리가 제대로 안 잡힌 것.

In [ ]:
import globpdbs = [f for f in glob.glob('*.pdb') if '_renum' not in f]print(f'대상 {len(pdbs)}개\n')!python renumber_site.py {' '.join(pdbs)}

## 5. 채점**읽는 법:**- `metal_fit_residual` — 배위 거리 오차(Å). 작을수록 좋음. 0.35 이하 통과- `angle_rmsd` — 사면체에서 벗어난 각도. 20° 이하 통과- `water_clash` — 촉매 물 자리를 막는 원자 수. **0이어야 함**- `escape_frac` — 용매로 나가는 통로 비율. 높을수록 열려 있음- `shell_hbond_n` — 2차 배위권 후보 수- `pass` — 위 조건 전부 통과했나

In [ ]:
!python metal_site_score.py "*_renum.pdb" --site zn_site.json --out scores.csv

## 6. 결과 표로 보기

In [ ]:
import pandas as pddf = pd.read_csv('scores.csv')df = df.sort_values('metal_fit_residual')cols = [c for c in ['pdb','metal_fit_residual','angle_rmsd','water_clash',                    'water_blocker','escape_frac','shell_hbond_n','pass'] if c in df.columns]df[cols]

## 7. 결과 내려받기

In [ ]:
from google.colab import filesfiles.download('scores.csv')

---## 문제가 생기면| 증상 | 원인 ||---|---|| `No such file or directory` | 2번에서 그 파일을 안 올림. 2번 다시 실행 || `배위원자 추출 실패` | 잔기 번호가 94/96/119가 아님. 4번을 먼저 돌렸는지 확인 || `JSONDecodeError` | `zn_site.json`이 진짜 JSON이 아님. 파일 내용 확인 || 표가 비어 있음 | `*_renum.pdb`가 안 만들어짐. 4번 출력 확인 |세션이 끊기면 올린 파일이 다 사라짐. 2번부터 다시 하면 됨.